# 🇳🇬 N-ATLaS Engine Replica: Serverless LLM & Sovereign ASR
### Google Colab (Zero Sign-up Tunneling)

This notebook provides a complete standalone replication of the **N-ATLaS Engine** (`NCAIR1/N-ATLaS` 8B multilingual Llama-3 fine-tune and **Sovereign ASR** models: Yoruba, Hausa, Igbo, and Nigerian Accented English).

**⚡ Zero Sign-In Required:**
- **Gradio UI** uses native `demo.launch(share=True)` — **No account, no sign-in, no API token needed!**
- **OpenAI-Compatible API** exposes both LLM (`/v1/chat/completions`) and ASR (`/v1/audio/transcriptions`) over a free Cloudflare Quick Tunnel so you can call it from Python/JS SDKs without creating any accounts.

## 1. Setup & Environment Verification
Check GPU availability and install dependencies.

In [1]:
# Check GPU accelerator
!nvidia-smi

# Install high-performance libraries (latest bitsandbytes for transformers 4.45+)
!pip install -q --upgrade pip
!pip install -q -U "bitsandbytes>=0.46.1"
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    gradio>=5.0.0 \
    fastapi>=0.115.0 \
    uvicorn>=0.30.0 \
    soundfile>=0.12.1 \
    librosa>=0.10.0 \
    pydantic>=2.8.0 \
    python-multipart>=0.0.12


Mon Oct  5 08:49:39 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Model Configuration
Define N-ATLaS and Sovereign ASR model IDs.

In [2]:
import os
import torch

MODEL_ID = "NCAIR1/N-ATLaS"
ASR_MODELS = {
    "yo": "NCAIR1/Yoruba-ASR",
    "yoruba": "NCAIR1/Yoruba-ASR",
    "ha": "NCAIR1/Hausa-ASR",
    "hausa": "NCAIR1/Hausa-ASR",
    "ig": "NCAIR1/Igbo-ASR",
    "igbo": "NCAIR1/Igbo-ASR",
    "en": "NCAIR1/NigerianAccentedEnglish",
    "nigerian-english": "NCAIR1/NigerianAccentedEnglish",
}
DEFAULT_ASR_MODEL = "NCAIR1/NigerianAccentedEnglish"

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

CUDA Available: True
Device: Tesla T4
VRAM: 14.56 GB


## 3. Load N-ATLaS 8B with 4-bit Quantization
Using `bitsandbytes` 4-bit NormalFloat (NF4) fits the full 16GB Llama-3 8B model into ~5.5GB of GPU memory on free Colab T4 GPUs.

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

# Retrieve HF Token from Environment or Colab Secrets
hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGING_FACE_HUB_TOKEN')
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN') or userdata.get('HUGGING_FACE_HUB_TOKEN')
    except Exception:
        pass

print('Loading Tokenizer (with authenticated token)...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=hf_token, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print('Configuring 4-bit BitsAndBytes quantization...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading {MODEL_ID} weights...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map='auto' if torch.cuda.is_available() else 'cpu',
    trust_remote_code=True,
)
model.eval()
print('N-ATLaS 8B loaded successfully!')


Loading Tokenizer (with authenticated token)...


config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

N-ATLaS 8B loaded successfully!


## 4. Test Text Generation in Nigerian Languages
Prompt testing across Hausa, Yoruba, Igbo, Nigerian Pidgin, and English.

In [5]:
def natlas_chat(
    prompt: str,
    system_prompt: str = "Your name is AwaGPT. You are a helpful Nigerian multilingual AI assistant.",
    max_new_tokens: int = 256,
    temperature: float = 0.7,
    repetition_penalty: float = 1.12,
):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    # Suppress BPE tokenizer warning by explicitly passing clean_up_tokenization_spaces
    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 0.01),
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id,
        )

    output_tokens = outputs[0][inputs.input_ids.shape[-1]:]
    return tokenizer.decode(output_tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()

# Test Prompts
tests = [
    ("Yoruba", "Bawo ni! Ki ni pataki imo-ero ayelujara si idagbasoke eto-eko?"),
    ("Hausa", "Sannu! Menene amfanin fasahar zamani wajen bunkasa ilimi?"),
    ("Igbo", "Kedu otu teknụzụ nwere ike isi nyere ụmụ akwụkwọ aka ịmụta ihe ọhụrụ?"),
    ("English", "Hello! What is the best way for someone to start a career in tech in Nigeria?"),
]

for lang, text in tests:
    print(f"\n--- [{lang}] Prompt: {text} ---")
    print(natlas_chat(text, max_new_tokens=150))



--- [Yoruba] Prompt: Bawo ni! Ki ni pataki imo-ero ayelujara si idagbasoke eto-eko? ---
Bawo! Imọ ero ayelujara, eyiti o kan iṣẹ-ṣiṣe ti iṣiro ati ikanni alaye, ṣe pataki fun idagbasoke eto-ọrọ nitori pe o gba awọn eniyan laaye lati wakọ awọn ọna asopọ, pinpin data, ati mu ibaraẹnisọrọ pọ si. Eyi le ja si ọpọlọpọ awọn anfani bii ilosoke ninu ifowosowopo, ipinnu to dara julọ da lori data didara, ati iranwo

--- [Hausa] Prompt: Sannu! Menene amfanin fasahar zamani wajen bunkasa ilimi? ---
Fasaha tana da matukar muhimmanci wajen inganta ilimi ta hanyoyi daban-daban, kamar yadda:

1. **Bude damammaki**: Intanet yana ba wa mutane samun bayanan ilimi marasa iyaka, suna baiwa malamai da dalibai damammaki don koyo da neman ilimi.

2. **Ingantacciyar sadarwa**: Fasaha tana sauƙaƙe haɗawa tsakanin malamai, ɗalibai, da iyalai, yana sauƙaƙe raba fahimta da

--- [Igbo] Prompt: Kedu otu teknụzụ nwere ike isi nyere ụmụ akwụkwọ aka ịmụta ihe ọhụrụ? ---
Enwere ọtụtụ teknụzụ dị mma nke nwere ike inyere

## 5. Load Sovereign Whisper ASR Pipeline
Transcribe audio clips using fine-tuned Whisper Small sovereign models.

In [6]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import io
import soundfile as sf
import librosa
import numpy as np

asr_pipelines = {}

def get_asr(model_name="yoruba"):
    repo = ASR_MODELS.get(model_name.lower(), DEFAULT_ASR_MODEL)
    if repo not in asr_pipelines:
        print(f"Loading ASR: {repo}...")
        processor = WhisperProcessor.from_pretrained(repo)
        asr_model = WhisperForConditionalGeneration.from_pretrained(
            repo,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        ).to("cuda" if torch.cuda.is_available() else "cpu")
        asr_model.eval()
        asr_pipelines[repo] = (asr_model, processor)
    return asr_pipelines[repo]

def transcribe_audio_bytes(audio_bytes, model_name="yoruba"):
    asr_model, processor = get_asr(model_name)
    try:
        audio_array, sr = sf.read(io.BytesIO(audio_bytes))
    except Exception:
        audio_array, sr = librosa.load(io.BytesIO(audio_bytes), sr=16000)
    if audio_array.ndim > 1:
        audio_array = np.mean(audio_array, axis=1)
    if sr != 16000:
        audio_array = librosa.resample(audio_array.astype(np.float32), orig_sr=sr, target_sr=16000)
        sr = 16000
    duration_sec = float(len(audio_array)) / 16000.0
    input_features = processor(audio_array, sampling_rate=16000, return_tensors="pt").input_features
    input_features = input_features.to(asr_model.device, dtype=asr_model.dtype)
    with torch.inference_mode():
        predicted_ids = asr_model.generate(input_features)
    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
    return {"text": text, "duration": round(duration_sec, 2), "model": ASR_MODELS.get(model_name.lower(), DEFAULT_ASR_MODEL)}

print("✅ Sovereign ASR Helper initialized!")

✅ Sovereign ASR Helper initialized!


## 6. Launch Interactive Gradio Web UI (Zero Sign-In Public Link)
`demo.launch(share=True)` gives you a live **`https://xxxx.gradio.live`** URL directly in the cell output with **zero sign-in required**.

In [8]:
import gradio as gr
from transformers import TextIteratorStreamer
import threading

def stream_chat(message, history, system_prompt, temperature, max_tokens, repetition_penalty):
    messages = [{"role": "system", "content": system_prompt or "You are AwaGPT, a Nigerian multilingual AI."}]
    for item in history:
        if isinstance(item, dict):
            messages.append(item)
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            if item[0]: messages.append({"role": "user", "content": str(item[0])})
            if item[1]: messages.append({"role": "assistant", "content": str(item[1])})
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

    gen_kwargs = dict(
        **inputs,
        streamer=streamer,
        max_new_tokens=int(max_tokens),
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    thread = threading.Thread(target=model.generate, kwargs=gen_kwargs)
    thread.start()

    partial = ""
    for chunk in streamer:
        partial += chunk
        yield partial

with gr.Blocks(title="N-ATLaS Colab Replica") as demo:
    gr.Markdown("# 🇳🇬 N-ATLaS Engine Replica (Colab Edition)")
    gr.Markdown("**Model:** `NCAIR1/N-ATLaS` (Llama-3 8B Multilingual) + **Sovereign ASR**")

    with gr.Tab("💬 Chatbot"):
        with gr.Row():
            with gr.Column(scale=3):
                chatbot = gr.Chatbot(height=450)
                msg = gr.Textbox(placeholder="Ask in Hausa, Yoruba, Igbo, or English...")
                btn = gr.Button("Send", variant="primary")
            with gr.Column(scale=1):
                sys_p = gr.Textbox(label="System Prompt", value="You are AwaGPT, a polite multilingual Nigerian assistant.")
                t = gr.Slider(0.0, 1.2, value=0.7, label="Temperature")
                m = gr.Slider(64, 2048, value=512, step=64, label="Max Tokens")
                r = gr.Slider(1.0, 1.4, value=1.12, step=0.02, label="Repetition Penalty")

        def user_turn(u_msg, hist):
            return "", hist + [{"role": "user", "content": u_msg}]

        def bot_turn(hist, s, temp, max_t, rep):
            last_msg = hist[-1]["content"]
            hist.append({"role": "assistant", "content": ""})
            for text in stream_chat(last_msg, hist[:-2], s, temp, max_t, rep):
                hist[-1]["content"] = text
                yield hist

        msg.submit(user_turn, [msg, chatbot], [msg, chatbot], queue=False).then(bot_turn, [chatbot, sys_p, t, m, r], chatbot)
        btn.click(user_turn, [msg, chatbot], [msg, chatbot], queue=False).then(bot_turn, [chatbot, sys_p, t, m, r], chatbot)

    with gr.Tab("🎙️ Speech-to-Text ASR"):
        audio_in = gr.Audio(sources=["microphone", "upload"], type="filepath")
        lang_choice = gr.Radio(["yoruba", "hausa", "igbo", "nigerian-english"], value="yoruba", label="Language")
        asr_btn = gr.Button("Transcribe Voice", variant="primary")
        asr_out = gr.Markdown()

        def on_transcribe(audio_path, lang):
            if not audio_path:
                return "Please provide audio input."
            with open(audio_path, "rb") as f:
                res = transcribe_audio_bytes(f.read(), lang)
            return f"### Transcript ({res['duration']}s):\n\n> **{res['text']}**"

        asr_btn.click(on_transcribe, inputs=[audio_in, lang_choice], outputs=asr_out)

# Zero sign-in public link generation
demo.queue().launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ab75b71aa4b26fb1d9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 7. Zero Sign-In OpenAI-Compatible API Gateway (LLM + ASR)
Run FastAPI on port 8000 and expose it instantly using **Cloudflare Quick Tunnel (`cloudflared`)** — **NO SIGN-IN, NO ACCOUNT, NO AUTH TOKEN NEEDED!**

Exposes the **exact same endpoints as N-ATLaS**:
- `GET  /healthz`
- `GET  /v1/models`
- `POST /v1/chat/completions` (LLM chat)
- `POST /v1/completions` (Raw completion)
- `POST /v1/audio/transcriptions` (Sovereign ASR)

Once running, you can connect with `OpenAI(base_url="<URL>/v1", api_key="none")` directly.

In [10]:
import subprocess, time, re, threading, uvicorn
from google.colab import userdata

# Start API server in background thread if not already running
def run_server():
    uvicorn.run(api_app, host="127.0.0.1", port=8000, log_level="warning")

try:
    if not server_thread.is_alive():
        server_thread = threading.Thread(target=run_server, daemon=True)
        server_thread.start()
except NameError:
    server_thread = threading.Thread(target=run_server, daemon=True)
    server_thread.start()
time.sleep(2)

print("Starting zero-sign-in Cloudflare Tunnel (No account needed)...\n")
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Robust extraction
public_url = None
for _ in range(50):
    line = proc.stderr.readline()
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        print(f"\n🚀 ZERO SIGN-IN PUBLIC API BASE URL:")
        print(f"   {public_url}/v1")
        print(f"\n👉 You can use this directly in Python:")
        print(f"   client = OpenAI(base_url='{public_url}/v1', api_key='none')")
        break


Starting zero-sign-in Cloudflare Tunnel (No account needed)...


🚀 ZERO SIGN-IN PUBLIC API BASE URL:
   https://vernon-trout-fame-polyphonic.trycloudflare.com/v1

👉 You can use this directly in Python:
   client = OpenAI(base_url='https://vernon-trout-fame-polyphonic.trycloudflare.com/v1', api_key='none')


In [12]:
from openai import OpenAI

# Plug in the exact URL you got
client = OpenAI(
    base_url='https://vernon-trout-fame-polyphonic.trycloudflare.com/v1',
    api_key='none'
)

response = client.chat.completions.create(
    model="NCAIR1/N-ATLaS",
    messages=[{"role": "user", "content": "Hello! what is AI?"}]
)

print(response.choices[0].message.content)


AI stands for Artificial Intelligence, which refers to the simulation of human intelligence in machines that are programmed to think and learn like humans.
